In [16]:
using Revise

In [4]:
using GeneRegulatorySystems
using GeneRegulatorySystems.Models
using GeneRegulatorySystems.Models.NetworkRepresentation
using GeneRegulatorySystems.Models.V1
using GeneRegulatorySystems.Scheduling
import JSON
using GarishPrint

macro pp(x) :(GarishPrint.pprint($(esc(x)))) end;

In [12]:

# Load ACDC (has proteolysis) and inspect current heuristic behaviour
using Catalyst

schedule_acdc = Models.load("examples/toy/ACDC.schedule.json", seed="seed123")

model_path = nothing
schedule_acdc(Models.FlatState(); dryrun = (primitive!, x, dt; path, _...) -> begin
    if isfinite(dt) && dt > 0.0 && model_path === nothing
        global model_path = primitive!.path
    end
end)
println("Model path: $model_path")

wrapped = Scheduling.reify(schedule_acdc, model_path)
v1_def_loaded = wrapped.model.definition
rs_loaded = wrapped.model.model.definition

all_rxns_acdc_loaded = Catalyst.reactions(rs_loaded)
reg_filtered = [rxn for rxn in all_rxns_acdc_loaded if NetworkRepresentation._is_regulation_reaction(rxn)]
non_reg      = [rxn for rxn in all_rxns_acdc_loaded if !NetworkRepresentation._is_regulation_reaction(rxn)]

println("Total reactions:       $(length(all_rxns_acdc_loaded))")
println("Heuristic-filtered:    $(length(reg_filtered))")
println("Surviving (heuristic): $(length(non_reg))")

println("\n=== Heuristic-filtered reactions ===")
for rxn in reg_filtered
    id = NetworkRepresentation._reaction_id(rxn)
    subs = isempty(rxn.substrates) ? "(none)" : join([String(NetworkRepresentation.SpeciesId(s).name) for s in rxn.substrates], " + ")
    prods = isempty(rxn.products) ? "(none)" : join([String(NetworkRepresentation.SpeciesId(p).name) for p in rxn.products], " + ")
    println("  $subs -> $prods  [id=$id]")
end


Model path: -2-1/1.do
Total reactions:       30
Heuristic-filtered:    6
Surviving (heuristic): 24

=== Heuristic-filtered reactions ===
  1.active -> (none)  [id=[1]1.active->]
  (none) -> 1.active  [id=->[1]1.active]
  2.active -> (none)  [id=[1]2.active->]
  (none) -> 2.active  [id=->[1]2.active]
  3.active -> (none)  [id=[1]3.active->]
  (none) -> 3.active  [id=->[1]3.active]


Load schedule

In [3]:
path_diff = "examples/specification/differentiation.schedule.json"
path_v1 = "examples/toy/ACDC.schedule.json"
path_kronecker = "examples/benchmark/kronecker-small.schedule.json"
path_rand_diff = "examples/specification/random-differentiation.schedule.json"

schedule! = Models.load(path_v1, seed="seed123");

In [ ]:

# Dryrun ACDC: inspect what segments are generated and their execution paths
sched_acdc = Models.load("examples/toy/ACDC.schedule.json", seed="seed123")

segments = []
sched_acdc(Models.FlatState(); dryrun = (primitive!, x, Δt; path, _...) -> begin
    push!(segments, (
        execution_path = path,
        model_path = primitive!.path,
        from = x.t,
        to = x.t + (isfinite(Δt) ? Δt : 0.0),
        is_instant = !isfinite(Δt) || Δt == 0.0,
    ))
end)

println("Total segments: $(length(segments))\n")
println("$(rpad("execution_path", 30))  $(rpad("model_path", 40))  from -> to")
println("-"^90)
for s in segments
    println("$(rpad(s.execution_path, 30))  $(rpad(s.model_path, 40))  $(s.from) -> $(s.to)$(s.is_instant ? "  [instant]" : "")")
end


In [ ]:
f! = Scheduling.reify(schedule!, "-2-1+-1+")

In [ ]:
rs = f!.f!.model.model.definition

Reify the schedule. Need to do a dryrun to get all the primitives

In [ ]:
# this only extracts the first network that it finds in the schedule, rather than all of them
function extract_network(schedule::Scheduling.Schedule)
    network = nothing
    
    function dryrun_collector(primitive!, x, Δt; path, _...)
        if !(isfinite(Δt) && Δt > 0.0)
            return
        end
        if network === nothing
            network = Models.NetworkRepresentation.entity(primitive!)
        end
    end
    
    schedule(Models.FlatState(); dryrun=dryrun_collector)
    return network
end

network = extract_network(schedule!)

# Reaction filtering exploration

Goal: find a principled way to identify which Catalyst reactions are "implementation artifacts" of explicit regulatory links (activation, repression, proteolysis) in a V1 definition, so we can filter exactly those rather than using a heuristic on species names.

Plan:
1. Load `copies.schedule.json` (has explicit `inactive` species + activation links)
2. Inspect both the V1 regulatory links AND the Catalyst reactions side by side
3. Understand what reactions V1 generates per link type
4. Derive a principled ID-based filter

In [7]:

# Load copies.schedule.json — has explicit inactive species + activation links
using Catalyst

sched_copies = Models.load("examples/specification/copies.schedule.json", seed="seed123")

wrapped_copies = nothing
sched_copies(Models.FlatState(); dryrun = (primitive!, x, dt; path, _...) -> begin
    if isfinite(dt) && dt > 0.0 && wrapped_copies === nothing
        global wrapped_copies = Scheduling.reify(sched_copies, primitive!.path)
    end
end)

v1_def = wrapped_copies.model.definition
rs = wrapped_copies.model.model.definition

println("Genes: ", [g.name for g in v1_def.genes])
println()

# Show all V1 regulatory links
desc = Models.describe(v1_def)
components = Dict(typeof(d) => d for d in desc.descriptions)
raw_links = components[Models.Network].links
println("=== Regulatory links in V1 definition ===")
for lnk in raw_links
    println("  kind=$(lnk.kind)  from=$(lnk.from)  to=$(lnk.to)  props=$(lnk.properties)")
end


Genes: [Symbol("1")]

=== Regulatory links in V1 definition ===


In [8]:
# List ALL Catalyst reactions with their IDs to understand what V1 generates
all_rxns = Catalyst.reactions(rs)
println("=== All $(length(all_rxns)) Catalyst reactions ===")
for rxn in all_rxns
    subs = [String(NetworkRepresentation.SpeciesId(s).name) for s in rxn.substrates]
    prods = [String(NetworkRepresentation.SpeciesId(p).name) for p in rxn.products]
    rxn_id = NetworkRepresentation._reaction_id(rxn)
    filtered = NetworkRepresentation._is_regulation_reaction(rxn)
    marker = filtered ? " [FILTERED]" : ""
    println("  $(join(subs, " + ")) -> $(join(prods, " + "))$marker")
end

=== All 10 Catalyst reactions ===
  1.active -> 1.inactive [FILTERED]
  1.inactive -> 1.active [FILTERED]
  1.active + polymerases -> 1.active + 1.elongations
  1.elongations -> 1.premrnas + polymerases
  1.premrnas -> 1.mrnas
  1.mrnas + ribosomes -> 1.mrnas + 1.proteins + ribosomes
  1.elongations -> polymerases
  1.premrnas -> 
  1.mrnas -> 
  1.proteins + proteasomes -> proteasomes


In [9]:
# Principled approach: derive the Catalyst reaction IDs that V1 generates
# per regulatory link type, from reading the V1 source.
#
# ACTIVATION(from=A, to=B):
#   basal deactivation:  B.active → B.inactive   (or B.active → 0 if B.unique)
#   basal activation:    B.inactive → B.active    (or 0 → B.active if B.unique)
#
# REPRESSION(from=A, to=B): same basal pair as activation (same reactions, different rates)
#   The catalytic effect is encoded in the *rate* expression, not the reaction stoichiometry.
#   So activation and repression share identical stoichiometric reactions!
#
# PROTEOLYSIS(from=A, to=B):
#   normal:    A.proteins + B.proteins → A.proteins
#   self-loop: B.proteins → 0  (stoich [2]→[1], net: one copy removed)
#
# Key insight: activation and repression basal reactions have the same IDs
# regardless of which gene does the regulating. We need per-target IDs only.
# The regulator (from) does NOT appear in the stoichiometry of these reactions.

gene_lookup = Dict(g.name => g for g in v1_def.genes)

function regulatory_reaction_ids_v2(raw_links, gene_lookup)::Set{Symbol}
    ids = Set{Symbol}()
    targets_seen = Set{Symbol}()  # avoid duplicates (activation + repression hit same basal rxns)

    for lnk in raw_links
        to = lnk.to
        from = lnk.from

        if lnk.kind in (:activation, :repression)
            target_gene = get(gene_lookup, to, nothing)
            to in targets_seen && continue
            push!(targets_seen, to)
            if target_gene !== nothing && target_gene.unique
                # unique gene: no inactive species, products/substrates are nothing
                push!(ids, Symbol("[1]$(to).active->"))        # B.active → 0
                push!(ids, Symbol("->[1]$(to).active"))        # 0 → B.active
            else
                push!(ids, Symbol("[1]$(to).active->[1]$(to).inactive"))
                push!(ids, Symbol("[1]$(to).inactive->[1]$(to).active"))
            end

        elseif lnk.kind == :proteolysis
            if from == to
                # self-loop: B.proteins → 0 (stoich [2]→[1])
                push!(ids, Symbol("[2]$(to).proteins->[1]$(to).proteins"))
            else
                push!(ids, Symbol("[1]$(from).proteins;[1]$(to).proteins->[1]$(from).proteins"))
            end
        end
    end
    ids
end

gene_names = Set{Symbol}(g.name for g in v1_def.genes)
principled_ids_v2 = regulatory_reaction_ids_v2(raw_links, gene_lookup)

println("=== Principled filter v2: $(length(principled_ids_v2)) IDs ===")
for id in principled_ids_v2
    println("  $id")
end

println()
println("=== Reactions matched ===")
for rxn in all_rxns
    rxn_id = NetworkRepresentation._reaction_id(rxn)
    subs = [String(NetworkRepresentation.SpeciesId(s).name) for s in rxn.substrates]
    prods = [String(NetworkRepresentation.SpeciesId(p).name) for p in rxn.products]
    matched = rxn_id in principled_ids_v2
    println("  $(join(subs, " + ")) -> $(join(prods, " + "))  [$(matched ? "FILTERED" : "kept")]")
end

=== Principled filter v2: 0 IDs ===

=== Reactions matched ===
  1.active -> 1.inactive  [kept]
  1.inactive -> 1.active  [kept]
  1.active + polymerases -> 1.active + 1.elongations  [kept]
  1.elongations -> 1.premrnas + polymerases  [kept]
  1.premrnas -> 1.mrnas  [kept]
  1.mrnas + ribosomes -> 1.mrnas + 1.proteins + ribosomes  [kept]
  1.elongations -> polymerases  [kept]
  1.premrnas ->   [kept]
  1.mrnas ->   [kept]
  1.proteins + proteasomes -> proteasomes  [kept]


In [10]:
# Diagnostic: print exact _reaction_id output for ALL reactions in copies model
# so we can verify the ID format matches what we generate above
println("=== Exact reaction IDs in copies model ===")
for rxn in all_rxns
    id = NetworkRepresentation._reaction_id(rxn)
    subs = isempty(rxn.substrates) ? ["(none)"] : [String(NetworkRepresentation.SpeciesId(s).name) for s in rxn.substrates]
    prods = isempty(rxn.products) ? ["(none)"] : [String(NetworkRepresentation.SpeciesId(p).name) for p in rxn.products]
    println("  id=$id")
    println("    $(join(subs, " + ")) -> $(join(prods, " + "))")
end

println()
println("Gene unique flags:")
for g in v1_def.genes
    println("  $(g.name): unique=$(g.unique)")
end

=== Exact reaction IDs in copies model ===
  id=[1]1.active->[1]1.inactive
    1.active -> 1.inactive
  id=[1]1.inactive->[1]1.active
    1.inactive -> 1.active
  id=[1]1.active;[1]polymerases->[1]1.active;[1]1.elongations
    1.active + polymerases -> 1.active + 1.elongations
  id=[1]1.elongations->[1]1.premrnas;[1]polymerases
    1.elongations -> 1.premrnas + polymerases
  id=[1]1.premrnas->[1]1.mrnas
    1.premrnas -> 1.mrnas
  id=[1]1.mrnas;[1]ribosomes->[1]1.mrnas;[1]1.proteins;[1]ribosomes
    1.mrnas + ribosomes -> 1.mrnas + 1.proteins + ribosomes
  id=[1]1.elongations->[1]polymerases
    1.elongations -> polymerases
  id=[1]1.premrnas->
    1.premrnas -> (none)
  id=[1]1.mrnas->
    1.mrnas -> (none)
  id=[1]1.proteins;[1]proteasomes->[1]proteasomes
    1.proteins + proteasomes -> proteasomes

Gene unique flags:
  1: unique=false


In [13]:

# Cross-check: apply principled filter to ACDC (has proteolysis, unique=true genes)
# Principled approach should match what the old heuristic found.

v1_def_acdc = wrapped.model.definition
rs_acdc = wrapped.model.model.definition
desc_acdc = Models.describe(v1_def_acdc)
components_acdc = Dict(typeof(d) => d for d in desc_acdc.descriptions)
raw_links_acdc = components_acdc[Models.Network].links
gene_lookup_acdc = Dict(g.name => g for g in v1_def_acdc.genes)

principled_ids_acdc = regulatory_reaction_ids_v2(raw_links_acdc, gene_lookup_acdc)
all_rxns_acdc = Catalyst.reactions(rs_acdc)
old_heuristic_ids_acdc = Set{Symbol}(
    NetworkRepresentation._reaction_id(rxn)
    for rxn in all_rxns_acdc
    if NetworkRepresentation._is_regulation_reaction(rxn)
)

println("Principled filter IDs: $(length(principled_ids_acdc))")
println("Old heuristic IDs:     $(length(old_heuristic_ids_acdc))")
println()
println("In principled but not heuristic: ", setdiff(principled_ids_acdc, old_heuristic_ids_acdc))
println("In heuristic but not principled: ", setdiff(old_heuristic_ids_acdc, principled_ids_acdc))

matched_principled = [NetworkRepresentation._reaction_id(r) for r in all_rxns_acdc if NetworkRepresentation._reaction_id(r) in principled_ids_acdc]
matched_heuristic  = [NetworkRepresentation._reaction_id(r) for r in all_rxns_acdc if NetworkRepresentation._is_regulation_reaction(r)]
println()
println("Reactions matched by principled: $matched_principled")
println("Reactions matched by heuristic:  $matched_heuristic")


Principled filter IDs: 6
Old heuristic IDs:     6

In principled but not heuristic: Set{Symbol}()
In heuristic but not principled: Set{Symbol}()

Reactions matched by principled: [Symbol("[1]1.active->"), Symbol("->[1]1.active"), Symbol("[1]2.active->"), Symbol("->[1]2.active"), Symbol("[1]3.active->"), Symbol("->[1]3.active")]
Reactions matched by heuristic:  [Symbol("[1]1.active->"), Symbol("->[1]1.active"), Symbol("[1]2.active->"), Symbol("->[1]2.active"), Symbol("[1]3.active->"), Symbol("->[1]3.active")]


In [14]:

# Cross-check principled filter on multifate2-proteolytic (has actual proteolysis links)
sched_prot = Models.load("examples/toy/multifate2-proteolytic.schedule.json", seed="seed123")

wrapped_prot = nothing
sched_prot(Models.FlatState(); dryrun = (primitive!, x, dt; path, _...) -> begin
    if isfinite(dt) && dt > 0.0 && wrapped_prot === nothing
        global wrapped_prot = Scheduling.reify(sched_prot, primitive!.path)
    end
end)

v1_def_prot = wrapped_prot.model.definition
rs_prot = wrapped_prot.model.model.definition
desc_prot = Models.describe(v1_def_prot)
components_prot = Dict(typeof(d) => d for d in desc_prot.descriptions)
raw_links_prot = components_prot[Models.Network].links
gene_lookup_prot = Dict(g.name => g for g in v1_def_prot.genes)

println("=== Genes ===")
for g in v1_def_prot.genes
    println("  $(g.name)  unique=$(g.unique)")
end
println()
println("=== Regulatory links ===")
for lnk in raw_links_prot
    println("  kind=$(lnk.kind)  from=$(lnk.from)  to=$(lnk.to)")
end

principled_ids_prot = regulatory_reaction_ids_v2(raw_links_prot, gene_lookup_prot)
all_rxns_prot = Catalyst.reactions(rs_prot)
old_heuristic_ids_prot = Set{Symbol}(
    NetworkRepresentation._reaction_id(rxn)
    for rxn in all_rxns_prot
    if NetworkRepresentation._is_regulation_reaction(rxn)
)

println()
println("Principled filter IDs: $(length(principled_ids_prot))")
println("Old heuristic IDs:     $(length(old_heuristic_ids_prot))")
println()
println("In principled but not heuristic: ", setdiff(principled_ids_prot, old_heuristic_ids_prot))
println("In heuristic but not principled: ", setdiff(old_heuristic_ids_prot, principled_ids_prot))

println()
println("=== All reactions ===")
for rxn in all_rxns_prot
    id = NetworkRepresentation._reaction_id(rxn)
    subs = isempty(rxn.substrates) ? "(none)" : join([String(NetworkRepresentation.SpeciesId(s).name) for s in rxn.substrates], " + ")
    prods = isempty(rxn.products) ? "(none)" : join([String(NetworkRepresentation.SpeciesId(p).name) for p in rxn.products], " + ")
    p = id in principled_ids_prot ? "P" : " "
    h = id in old_heuristic_ids_prot ? "H" : " "
    println("  [$p$h] $subs -> $prods")
end
println("(P=principled, H=heuristic)")


=== Genes ===
  1  unique=true
  2  unique=true
  3  unique=true

=== Regulatory links ===
  kind=activation  from=2  to=2
  kind=proteolysis  from=1  to=2
  kind=proteolysis  from=3  to=2
  kind=activation  from=3  to=3
  kind=proteolysis  from=1  to=3
  kind=proteolysis  from=2  to=3

Principled filter IDs: 8
Old heuristic IDs:     6

In principled but not heuristic: Set([Symbol("[1]1.proteins;[1]3.proteins->[1]1.proteins"), Symbol("[1]1.proteins;[1]2.proteins->[1]1.proteins"), Symbol("[1]2.proteins;[1]3.proteins->[1]2.proteins"), Symbol("[1]3.proteins;[1]2.proteins->[1]3.proteins")])
In heuristic but not principled: Set([Symbol("->[1]1.active"), Symbol("[1]1.active->")])

=== All reactions ===
  [ H] 1.active -> (none)
  [ H] (none) -> 1.active
  [PH] 2.active -> (none)
  [PH] (none) -> 2.active
  [P ] 1.proteins + 2.proteins -> 1.proteins
  [P ] 3.proteins + 2.proteins -> 3.proteins
  [PH] 3.active -> (none)
  [PH] (none) -> 3.active
  [P ] 1.proteins + 3.proteins -> 1.proteins
  [

In [15]:

# Cross-check principled filter on multifate2-dimer
sched_dimer = Models.load("examples/toy/multifate2-dimer.schedule.json", seed="seed123")

wrapped_dimer = nothing
sched_dimer(Models.FlatState(); dryrun = (primitive!, x, dt; path, _...) -> begin
    if isfinite(dt) && dt > 0.0 && wrapped_dimer === nothing
        global wrapped_dimer = Scheduling.reify(sched_dimer, primitive!.path)
    end
end)

v1_def_dimer = wrapped_dimer.model.definition
rs_dimer = wrapped_dimer.model.model.definition
desc_dimer = Models.describe(v1_def_dimer)
components_dimer = Dict(typeof(d) => d for d in desc_dimer.descriptions)
raw_links_dimer = components_dimer[Models.Network].links
gene_lookup_dimer = Dict(g.name => g for g in v1_def_dimer.genes)

println("=== Genes ===")
for g in v1_def_dimer.genes
    println("  $(g.name)  unique=$(g.unique)")
end
println()
println("=== Regulatory links ===")
for lnk in raw_links_dimer
    println("  kind=$(lnk.kind)  from=$(lnk.from)  to=$(lnk.to)")
end

principled_ids_dimer = regulatory_reaction_ids_v2(raw_links_dimer, gene_lookup_dimer)
all_rxns_dimer = Catalyst.reactions(rs_dimer)
old_heuristic_ids_dimer = Set{Symbol}(
    NetworkRepresentation._reaction_id(rxn)
    for rxn in all_rxns_dimer
    if NetworkRepresentation._is_regulation_reaction(rxn)
)

println()
println("Principled filter IDs: $(length(principled_ids_dimer))")
println("Old heuristic IDs:     $(length(old_heuristic_ids_dimer))")
println()
println("In principled but not heuristic: ", setdiff(principled_ids_dimer, old_heuristic_ids_dimer))
println("In heuristic but not principled: ", setdiff(old_heuristic_ids_dimer, principled_ids_dimer))

println()
println("=== All reactions ===")
for rxn in all_rxns_dimer
    id = NetworkRepresentation._reaction_id(rxn)
    subs = isempty(rxn.substrates) ? "(none)" : join([String(NetworkRepresentation.SpeciesId(s).name) for s in rxn.substrates], " + ")
    prods = isempty(rxn.products) ? "(none)" : join([String(NetworkRepresentation.SpeciesId(p).name) for p in rxn.products], " + ")
    p = id in principled_ids_dimer ? "P" : " "
    h = id in old_heuristic_ids_dimer ? "H" : " "
    println("  [$p$h] $subs -> $prods")
end
println("(P=principled, H=heuristic)")


=== Genes ===
  A  unique=true
  B  unique=true

=== Regulatory links ===
  kind=activation  from=AA  to=A
  kind=activation  from=BB  to=B

Principled filter IDs: 4
Old heuristic IDs:     4

In principled but not heuristic: Set{Symbol}()
In heuristic but not principled: Set{Symbol}()

=== All reactions ===
  [PH] A.active -> (none)
  [PH] (none) -> A.active
  [PH] B.active -> (none)
  [PH] (none) -> B.active
  [  ] A.proteins -> AA
  [  ] A.proteins + B.proteins -> AB
  [  ] B.proteins -> BB
  [  ] AA -> A.proteins
  [  ] AB -> A.proteins + B.proteins
  [  ] BB -> B.proteins
  [  ] A.active + polymerases -> A.active + A.elongations
  [  ] A.elongations -> A.premrnas + polymerases
  [  ] A.premrnas -> A.mrnas
  [  ] A.mrnas + ribosomes -> A.mrnas + A.proteins + ribosomes
  [  ] A.elongations -> polymerases
  [  ] A.premrnas -> (none)
  [  ] A.mrnas -> (none)
  [  ] A.proteins + proteasomes -> proteasomes
  [  ] B.active + polymerases -> B.active + B.elongations
  [  ] B.elongations -> 

In [17]:

# Smoke-test the new production code path (entity via Wrapped)
# Tests copies (unique=false, no links), multifate2-proteolytic (proteolysis), multifate2-dimer

function test_entity(label, wrapped)
    e = NetworkRepresentation.entity(wrapped)
    nodes, links = NetworkRepresentation.flatten(e)
    n_genes    = count(n -> n.kind == :gene,     nodes)
    n_species  = count(n -> n.kind == :species,  nodes)
    n_reactions = count(n -> n.kind == :reaction, nodes)
    println("$label: $n_genes genes, $n_species species, $n_reactions reactions")
end

test_entity("copies          ", wrapped_copies)
test_entity("multifate2-prot ", wrapped_prot)
test_entity("multifate2-dimer", wrapped_dimer)
test_entity("ACDC            ", wrapped)


copies          : 1 genes, 9 species, 8 reactions
multifate2-prot : 3 genes, 18 species, 28 reactions
multifate2-dimer: 2 genes, 16 species, 22 reactions
ACDC            : 3 genes, 18 species, 24 reactions
